<a href="https://colab.research.google.com/github/heisenberg304/heart-disease-ml-classification/blob/main/RandomForest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Cargar librerias
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, precision_score, f1_score

#Cargar dataset
data = pd.read_csv("/content/drive/MyDrive/data_csv/heart_disease_uci.csv")

#===================================================================================================================================================================================================

#MANIPULAR CSV
  #rellenar: trestbps(59)(median o mean), chol(30)(median), fbs(90)(mode), restecg(2)(mode), thalch(55)(median o mean), oldpeak(62)(median)
data["trestbps"].fillna(data["trestbps"].median(), inplace=True)
data["chol"].fillna(data["chol"].median(), inplace=True)
data["fbs"].fillna(data["fbs"].mode()[0], inplace=True)
data["restecg"].fillna(data["restecg"].mode()[0], inplace=True)
data["thalch"].fillna(data["thalch"].median(), inplace=True)
data["oldpeak"].fillna(data["oldpeak"].median(), inplace=True)

  #target binario
data["num"] = data["num"].apply(lambda x: 1 if x > 0 else 0)

  #encoding: sex, cp, restecg, num (target)
fe_ma = pd.get_dummies(data["sex"])
CP = pd.get_dummies(data["cp"])
RESTECG = pd.get_dummies(data["restecg"])

  #union de datasets
data = pd.concat([data, fe_ma, CP, RESTECG], axis=1)
  #borrar columnas
data = data.drop(["id", "dataset", "slope", "ca", "thal", "sex", "cp", "restecg"], axis=1)

#===================================================================================================================================================================================================

#crear x and y
x = data[['age','trestbps','chol','fbs','thalch','exang',
          'oldpeak','Female','Male','asymptomatic',
          'atypical angina','non-anginal','typical angina',
          'lv hypertrophy','normal','st-t abnormality']]
y = data["num"]

#split 80/20
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size = 0.2,
    random_state=55
)

#===================================================================================================================================================================================================

#crear modelo
model = RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=4)
#entrenar modelo
model.fit(x_train, y_train)
#prediccion
#pred = model.predict(x_test)
#threshold
probs = model.predict_proba(x_test)[:, 1]
threshold = 0.35
pred = (probs >= threshold).astype(int)

#Importancia de cada feature
importances = model.feature_importances_

#===================================================================================================================================================================================================

#MODEL PERFORMANCE

# -----confusion matrix--------
# |                           |
# |          Pred 0   Pred 1  |
# | Real 0     54       30    |
# | Real 1      6       94    |
# -----------------------------
#accuracy  -> 0.7989130434782609
#precision -> 0.752
#recall    -> 0.94
#f1 score  -> 0.8355555555555556
#Analisis:
#El modelo logra un recall alto (0.94), lo cual indica que detecta correctamente la mayoría de pacientes con enfermedad.
#Sin embargo, la precisión es menor (0.75), lo que implica falsos positivos.
#Este comportamiento es aceptable dado que en un contexto médico es preferible no dejar pasar casos enfermos.

#Los 3 Importance mas relevantes
#"asymptomatic" [0.22129586] El modelo detecta que la ausencia de síntomas típicos no implica ausencia de enfermedad, lo cual es clínicamente coherente.
#"oldpeak"      [0.12007416] Es una variable médica directa relacionada con problemas cardíacos, por lo que tiene fuerte poder predictivo.
#"exang"        [0.11460541] Indica que el corazón no responde bien al esfuerzo, lo que es un fuerte indicador clínico

#===================================================================================================================================================================================================

#metricas
matrix = confusion_matrix(y_test, pred)
accuracy = accuracy_score(y_test, pred)
precision = precision_score(y_test, pred)
recall = recall_score(y_test, pred)
f1 = f1_score(y_test, pred)
#print de metricas
print(f"matrix -> {matrix}")
print(f"accuracy -> {accuracy}")
print(f"precision -> {precision}")
print(f"recall -> {recall}")
print(f"f1 score -> {f1}")
print(f"Importancia = {importances}")


